# Datenquelle Chicago: Crimes – 2001 to Present

**Zweck:** Dieses Notebook dokumentiert die Chicago-Quelle, bevor wir Daten laden oder transformieren. Es beschreibt, woher die Daten stammen, wie sie technisch bereitgestellt werden und was eine Zeile fachlich bedeutet.

**Stand der Dokumentation:** 3. September 2026

## 1. Kurzüberblick

| Merkmal | Beschreibung |
|---|---|
| Datensatz | **Crimes – 2001 to Present** |
| Herausgeber | City of Chicago Data Portal |
| Datenverantwortlicher | Chicago Police Department (CPD) |
| Quellsystem | CLEAR – Citizen Law Enforcement Analysis and Reporting |
| Plattform | Socrata Open Data |
| Schnittstelle | SODA – Socrata Open Data API |
| Dataset-ID | `ijzp-q8t2` |
| Zeitraum | 2001 bis heute, ohne die jeweils jüngsten rund sieben Tage (am 19.9.2026 gemessen: 8 Tage) |
| Aktualisierung | täglich |
| Kategorie im Portal | Public Safety |
| Räumlicher Bezug | City of Chicago |

## 2. Um welche Art von Quelle handelt es sich?

Das Chicago Police Department extrahiert Daten aus der **operativen Polizeidatenbank CLEAR** und veröffentlicht sie im Open-Data-Portal der Stadt.

Dort liegt die Quelle als **tabellarischer Socrata-Datensatz** vor: fachlich ähnlich einer grossen Tabelle mit Zeilen und Spalten. Socrata stellt diese Tabelle zusätzlich über eine HTTP-API bereit. Die API heisst **SODA (Socrata Open Data API)** und kann Daten unter anderem als JSON, CSV oder GeoJSON liefern.

Die SODA API unterstützt mit **SoQL** eine SQL-ähnliche Abfragesprache. Dadurch wird es möglich bereits auf der Quellseite Spalten auswählen, Zeiträume und Deliktarten zu filtern, sortieren und aggregieren. Das reduziert unnötige Downloads.

## 3. Datensatz und fachliche Bedeutung einer Zeile

Der Datensatz beschreibt laut Chicago gemeldete Straftaten in der Stadt. Für die meisten Deliktarten steht eine Zeile grundsätzlich für einen **gemeldeten Crime Incident**. Die Spalte `id` identifiziert den Datensatz, während `case_number` die CPD Records Division Number des Vorfalls enthält.

### Sonderfall Murder/Homicide

Murder/Homicide ist nicht separat im Sinn von „im Crime-Dataset gar nicht vorhanden“. Chicago weist aber auf eine andere Granularität hin:

- Bei Murder/Homicide existiert ein Datensatz **pro Opfer**.
- Die übrigen Crimes werden grundsätzlich als gemeldete **Crime Incidents** geführt.

Damit besitzt `COUNT(*)` bei Homicide nicht die gleiche Bedeutung wie beispielsweise bei Robbery oder Burglary. Technisch sieht beides nach einer Zeile aus, fachlich repräsentiert die Zeile jedoch eine andere Einheit.

## 4. Kategorien im ersten PoC (LN1)

Alle Deliktskategorien in `primary_type` werden verarbeitet. Die Kategorien werden nicht als feste Liste im Code hinterlegt, weil sich Quellwerte und Klassifikationen im Lauf der Zeit ändern könnten.

Dabei gelten folgende Regeln:

- Der ursprüngliche Wert aus `primary_type` bleibt in **Bronze** unverändert erhalten. Silver führt ihn nicht weiter; dort steht Chicagos `description` als `source_offense_description`, der Quellcode `iucr` als `source_offense_code`.
- Die Vereinheitlichung für den Städtevergleich erfolgt in einer zusätzlichen Spalte (`nibrs_code`), mit `mapping_status` als Güteangabe je Zeile. Der Quellcode wird nicht überschrieben.
- Neue oder unbekannte Kategorien werden nicht verworfen: unerwartete Felder landen in `_extra`, nicht zuzuordnende Codes tragen `mapping_status = unmapped`.
- **Chicago führt bei Tötungsdelikten eine Zeile je Opfer.** Im Jahrgang 2024 sind das 597 Zeilen auf 568 Fälle. Alle 25 Chicagoer Fälle mit mehr als einer Zeile tragen NIBRS `09A`.

Über Kategorien hinweg darf deshalb nicht unbesehen `COUNT(*)` gerechnet werden. Silver löst das mit zwei Markierungen je Zeile: `counts` für die **Tatebene**, `counts_case` für die **Fallebene**. Auf der Tatebene zählen alle 597 Zeilen, auf der Fallebene 568.

> **Nachtrag 19.9.2026.** Hier stand bis heute, Homicide erhalte die Kennzeichnung `record_granularity = VICTIM` und die übrigen Kategorien `record_granularity = INCIDENT`. Ein solches Feld gibt es für Chicago nicht — weder als Spalte noch im Manifest. `record_granularity` existiert nur in Detroits Bronze-Manifest, als Angabe für den ganzen Lauf mit dem Wert `INCIDENT_OFFENSE`. Die Sache selbst ist richtig und gelöst, nur über `counts` und `counts_case`.


## 5. Wichtige Quellspalten

| Spalte | Fachliche Bedeutung | Voraussichtliche Nutzung |
|---|---|---|
| `id` | eindeutige Kennung des Datensatzes | technische Identifikation / Deduplizierung |
| `case_number` | CPD Records Division Number des Incidents | fachliche Referenz; Granularität prüfen |
| `date` | Zeitpunkt des Ereignisses; teilweise geschätzt | zeitliche Analyse |
| `block` | aus Datenschutzgründen teilweise anonymisierte Adresse | grobe räumliche Einordnung |
| `iucr` | Illinois Uniform Crime Reporting Code | detaillierte Deliktklassifikation |
| `primary_type` | primäre Beschreibung des IUCR-Codes | Übernahme aller vorhandenen Deliktkategorien |
| `description` | detailliertere Unterkategorie | optionale Detailanalyse |
| `location_description` | Art des Tatorts | optionale Kontextanalyse |
| `arrest` | ob eine Verhaftung erfolgte | optionales Analysemerkmal |
| `domestic` | Kennzeichnung eines häuslichen Bezugs | optionales Analysemerkmal |
| `district`, `ward`, `community_area` | administrative Gebiete | räumliche Aggregation |
| `year` | Jahr des Ereignisses | Filter und Partitionierung |
| `updated_on` | letzte Aktualisierung des Datensatzes | inkrementelle Verarbeitung / Nachvollziehbarkeit |
| `latitude`, `longitude`, `location` | approximative Geokoordinaten | Karten und räumliche Zuordnung |

Die vollständige Spaltenliste und die jeweils aktuellen Datentypen liefert der offizielle Metadaten-Endpunkt.

> **Nachtrag 19.9.2026 — hier ist die Umsetzung bewusst anders ausgefallen.** Geplant war, sich auf die benötigten Spalten zu beschränken. Die Ingestion fragt die API heute **ohne `$select`** ab und pinnt stattdessen alle 22 Quellfelder; was neu dazukommt, landet in `_extra` statt verlorenzugehen. Der Kommentar im Code sagt es so: *„The API is queried without `$select`, so whole records arrive and a new field lands in EXTRA_COLUMN rather than being lost."* Die Testabfrage unten wählt weiterhin nur die 18 wichtigen Spalten — sie zeigt die Quelle, sie lädt nicht.

## 6. Technischer API-Zugriff

### Identifikation

- Portal-Domain: `data.cityofchicago.org`
- Dataset-ID: `ijzp-q8t2`
- Metadaten: `https://data.cityofchicago.org/api/views/ijzp-q8t2`

### SODA-Endpunkte

Socrata dokumentiert derzeit einen SODA-3-Abfrage-Endpunkt:

`https://data.cityofchicago.org/api/v3/views/ijzp-q8t2/query.json`

Daneben gibt es den klassischen ressourcenorientierten SODA-Endpunkt, der sich besonders einfach mit HTTP GET und SoQL-Parametern verwenden lässt:

`https://data.cityofchicago.org/resource/ijzp-q8t2.json`

Vor der Implementierung entscheiden wir uns bewusst für **einen** Endpunkt und halten den Zugriff einfach. Grosse Resultate müssen seitenweise geladen werden; eine stabile Sortierung ist für reproduzierbares Paging wichtig.

> **Nachtrag 19.9.2026 — entschieden.** Die Ingestion benutzt den klassischen ressourcenorientierten Endpunkt `https://data.cityofchicago.org/resource/ijzp-q8t2.json`; er steht so in jedem Bronze-Manifest unter `query.url_first_page`. Abgefragt wird ein Zeitraum, aber **keine Spaltenauswahl** (siehe Abschnitt 5). Die Deliktkategorien werden nicht eingeschränkt. Paginiert wird nicht über `$offset`, sondern über eine vorab fixierte `id`-Liste — damit verschieben gleichzeitig eingefügte Zeilen keine Seiten.

### App-Token

Der Socrata App Token wird lokal als `SOCRATA_APP_TOKEN` in der Datei `.env` gespeichert. Bei SODA wird der Token bevorzugt im HTTP-Header `X-App-Token` übertragen. Ein App Token ordnet Anfragen einer Anwendung zu und verbessert die verfügbaren Request-Limits; er ist nicht automatisch eine Benutzeranmeldung.

In [8]:
# Zentrale, nicht geheime Kennwerte der Quelle
CHICAGO_SOURCE = {
    "domain": "data.cityofchicago.org",
    "dataset_id": "ijzp-q8t2",
    "dataset_name": "Crimes - 2001 to Present",
    "publisher": "City of Chicago",
    "attribution": "Chicago Police Department",
    "source_system": "CLEAR",
    "api": "SODA",
}

CHICAGO_SOURCE

{'domain': 'data.cityofchicago.org',
 'dataset_id': 'ijzp-q8t2',
 'dataset_name': 'Crimes - 2001 to Present',
 'publisher': 'City of Chicago',
 'attribution': 'Chicago Police Department',
 'source_system': 'CLEAR',
 'api': 'SODA'}

## 7. Datenqualität, Datenschutz und Grenzen

Die folgenden Einschränkungen stammen aus der offiziellen Datensatzbeschreibung und müssen bei jeder Interpretation berücksichtigt werden:

- Die jeweils **jüngsten sieben Tage fehlen**.
- Angaben können vorläufig sein, auf Meldungen der berichtenden Personen beruhen und noch nicht verifiziert sein.
- Deliktklassifikationen können sich durch spätere Ermittlungen ändern.
- Mechanische oder menschliche Fehler sind möglich; CPD garantiert weder Vollständigkeit noch Genauigkeit oder zeitlich korrekte Reihenfolge.
- CPD weist ausdrücklich darauf hin, dass die Daten nicht für Vergleiche über die Zeit verwendet werden sollen. Zeitreihen müssen deshalb besonders vorsichtig interpretiert und diese Einschränkung muss im Bericht genannt werden.
- Adressen werden zum Schutz von Opfern nur auf Blockebene angezeigt.
- Kartenpositionen sind approximativ; aus ihnen dürfen keine konkreten Adressen rekonstruiert werden.
- Der Datensatz kann nachträglich aktualisiert werden. Ein wiederholter Abruf desselben Zeitraums kann deshalb veränderte Werte liefern.

Diese Quelle bildet **gemeldete und im System erfasste Straftaten** ab. Sie misst daher nicht automatisch die gesamte tatsächlich begangene Kriminalität.

## 8. Entscheidungen für die Chicago-Ingestion

1. Wir lesen die Quelle über die offizielle SODA API.
2. Wir laden nicht sofort den vollständigen Datensatz seit 2001, sondern beginnen mit dem Jahr 2024.
3. Wir grenzen die Abfrage auf einen klaren Zeitraum ein und übernehmen alle Deliktkategorien. ~~Wir beschränken die Abfrage auf benötigte Spalten.~~ → **Verworfen**, siehe Abschnitt 5: Es wird ohne `$select` geladen, damit ein neues Feld in `_extra` landet statt verlorenzugehen.
4. Wir verwenden `id` als technische Quellkennung und prüfen `case_number` zusätzlich auf Mehrfachvorkommen. → **Geprüft und entschieden:** 25 Fallnummern kommen mehrfach vor, `case_number` taugt deshalb nicht als Deliktschlüssel. In Silver ist `id` der `offense_id`, `case_number` der `event_id`.
5. Wir dokumentieren Abrufzeitpunkt, Filter und verwendeten API-Endpunkt. → Steht je Lauf im Manifest, dazu `contract_version` und der Git-Commit.
6. Homicide bleibt enthalten und wird in Auswertungen separat behandelt — über die Markierungen `counts` (Tatebene) und `counts_case` (Fallebene), **nicht** über ein eigenes Granularitätsfeld.

> **Nachtrag 19.9.2026.** Punkt 6 lautete ursprünglich, Homicide werde „explizit mit Opfer-Granularität gekennzeichnet". Ein solches Feld gibt es nicht; siehe Abschnitt 4.


## 9. Einfache Testabfrage

Die folgende Abfrage lädt genau fünf Zeilen aus dem Jahr 2024 über den klassischen SODA-Endpunkt und zeigt das Ergebnis als Pandas DataFrame an. Der App Token wird aus der lokalen `.env` gelesen und im HTTP-Header gesendet. Er wird weder im Notebook noch in der Ausgabe angezeigt.

Mit `$select` wählen wir alle unter Abschnitt 5 dokumentierten wichtigen Quellspalten aus, mit `$where` grenzen wir den Zeitraum ein und mit `$limit` begrenzen wir die Antwort. Die Deliktkategorien werden **nicht gefiltert**.

In [9]:
import os

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()

selected_columns = [
    "id",
    "case_number",
    "date",
    "block",
    "iucr",
    "primary_type",
    "description",
    "location_description",
    "arrest",
    "domestic",
    "district",
    "ward",
    "community_area",
    "year",
    "updated_on",
    "latitude",
    "longitude",
    "location",
]

url = "https://data.cityofchicago.org/resource/ijzp-q8t2.json"
headers = {"X-App-Token": os.environ["SOCRATA_APP_TOKEN"]}
params = {
    "$select": ",".join(selected_columns),
    "$where": "date between '2024-01-01T00:00:00' and '2024-12-31T23:59:59'",
    "$order": "date,id",
    "$limit": 5,
}

response = requests.get(url, headers=headers, params=params, timeout=30)
response.raise_for_status()
records = response.json()
df = pd.DataFrame(records, columns=selected_columns)
df

,id,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,district,ward,community_area,year,updated_on,latitude,longitude,location
0,13324861,JH100083,2024-01-01T00:00:00.000,050XX W VAN BUREN ST,041A,BATTERY,AGGRAVATED - HANDGUN,ALLEY,False,False,015,29,25,2024,2024-12-21T15:40:46.000,41.874896437,-87.751134319,"{'latitude': '41.874896437', 'longitude': '-87..."
1,13324885,JH100099,2024-01-01T00:00:00.000,034XX W DRUMMOND PL,0486,BATTERY,DOMESTIC BATTERY SIMPLE,APARTMENT,True,True,014,35,22,2024,2024-12-21T15:40:46.000,41.9292756,-87.713262556,"{'latitude': '41.9292756', 'longitude': '-87.7..."
2,13324959,JH100203,2024-01-01T00:00:00.000,024XX N MILWAUKEE AVE,0530,ASSAULT,AGGRAVATED - OTHER DANGEROUS WEAPON,STREET,False,True,014,1,22,2024,2024-12-21T15:40:46.000,41.925072551,-87.700886541,"{'latitude': '41.925072551', 'longitude': '-87..."
3,13325121,JH100358,2024-01-01T00:00:00.000,122XX S SANGAMON ST,0486,BATTERY,DOMESTIC BATTERY SIMPLE,RESIDENCE,False,True,005,21,53,2024,2024-12-21T15:40:46.000,41.671722201,-87.64536182,"{'latitude': '41.671722201', 'longitude': '-87..."
4,13325146,JH100483,2024-01-01T00:00:00.000,077XX S LUELLA AVE,1320,CRIMINAL DAMAGE,TO VEHICLE,STREET,False,False,004,7,43,2024,2024-12-21T15:40:46.000,41.754516844,-87.569944944,"{'latitude': '41.754516844', 'longitude': '-87..."


## 10. Offizielle Quellen

- [Chicago Data Portal: Crimes – 2001 to Present](https://data.cityofchicago.org/Public-Safety/Crimes-2001-to-Present/ijzp-q8t2/about_data)
- [Offizielle Metadaten des Datensatzes](https://data.cityofchicago.org/api/views/ijzp-q8t2)
- [Socrata API Foundry für `ijzp-q8t2`](https://dev.socrata.com/foundry/data.cityofchicago.org/ijzp-q8t2)
- [SODA Consumer API – Einstieg](https://dev.socrata.com/consumers/getting-started)
- [SODA-API-Endpunkte](https://dev.socrata.com/docs/endpoints)
- [SoQL-Abfragen mit SODA 3](https://dev.socrata.com/docs/queries/)
- [Socrata Application Tokens](https://dev.socrata.com/docs/app-tokens.html)
- [Chicago Police Department IUCR Codes](https://data.cityofchicago.org/Public-Safety/Chicago-Police-Department-Illinois-Uniform-Crime-R/c7ck-438e)

Die fachlichen Aussagen zur Abdeckung, Aktualisierung, Granularität, Herkunft aus CLEAR und zu den Einschränkungen basieren auf der offiziellen Datensatzbeschreibung. Technische Aussagen zur API basieren auf der offiziellen Socrata-Dokumentation.

# Appendix A: Vollständiges Quellschema

Die folgende Tabelle enthält alle 31 Felder, die der offizielle Metadaten-Endpunkt am 3. September 2026 für den Datensatz `ijzp-q8t2` ausweist. **Ja** bedeutet, dass das Feld in Abschnitt 5 als wichtig eingestuft wurde und in der aktuellen Testabfrage enthalten ist. **Nein** bedeutet nur, dass das Feld für den ersten PoC nicht ausgewählt wurde; es bleibt weiterhin Bestandteil der Quelle.

| Pos. | API-Feld | Portalname | Socrata-Typ | Wichtig | Beschreibung |
|---:|---|---|---|:---:|---|
| 1 | `id` | ID | number | **Ja** | Eindeutige Kennung des Datensatzes. |
| 2 | `case_number` | Case Number | text | **Ja** | CPD Records Division Number des Incidents. Bei opferbezogenen Homicide-Datensätzen muss die mögliche Mehrfachverwendung fachlich geprüft werden. |
| 3 | `date` | Date | calendar_date | **Ja** | Zeitpunkt, zu dem der Incident stattfand; teilweise handelt es sich um eine Schätzung. |
| 4 | `block` | Block | text | **Ja** | Teilweise anonymisierte Adresse, die nur den Block des tatsächlichen Orts zeigt. |
| 5 | `iucr` | IUCR | text | **Ja** | Illinois Uniform Crime Reporting Code; direkt mit `primary_type` und `description` verbunden. |
| 6 | `primary_type` | Primary Type | text | **Ja** | Primäre Beschreibung des IUCR-Codes und zentrale Deliktkategorie. |
| 7 | `description` | Description | text | **Ja** | Sekundäre Beschreibung beziehungsweise Unterkategorie des IUCR-Codes. |
| 8 | `location_description` | Location Description | text | **Ja** | Beschreibung der Art des Orts, an dem der Incident stattfand. |
| 9 | `arrest` | Arrest | checkbox | **Ja** | Gibt an, ob eine Verhaftung vorgenommen wurde. |
| 10 | `domestic` | Domestic | checkbox | **Ja** | Gibt an, ob der Incident gemäss Illinois Domestic Violence Act einen häuslichen Bezug hat. |
| 11 | `beat` | Beat | text | Nein | Kleinste geografische Polizeieinheit; mehrere Beats bilden einen Sektor und mehrere Sektoren einen District. |
| 12 | `district` | District | text | **Ja** | Polizeidistrikt, in dem der Incident stattfand. |
| 13 | `ward` | Ward | number | **Ja** | Politischer Stadtbezirk beziehungsweise City Council District des Incidents. |
| 14 | `community_area` | Community Area | text | **Ja** | Community Area des Incidents; Chicago besitzt 77 Community Areas. |
| 15 | `fbi_code` | FBI Code | text | Nein | Deliktklassifikation nach dem National Incident-Based Reporting System (NIBRS) des FBI. |
| 16 | `x_coordinate` | X Coordinate | number | Nein | Verschobene X-Koordinate in der Projektion State Plane Illinois East NAD 1983; liegt nur auf demselben Block wie der tatsächliche Ort. |
| 17 | `y_coordinate` | Y Coordinate | number | Nein | Verschobene Y-Koordinate in der Projektion State Plane Illinois East NAD 1983; liegt nur auf demselben Block wie der tatsächliche Ort. |
| 18 | `year` | Year | number | **Ja** | Jahr, in dem der Incident stattfand. |
| 19 | `updated_on` | Updated On | calendar_date | **Ja** | Datum und Uhrzeit der letzten Aktualisierung des Datensatzes. |
| 20 | `latitude` | Latitude | number | **Ja** | Aus Datenschutzgründen verschobener Breitengrad; liegt auf demselben Block wie der tatsächliche Ort. |
| 21 | `longitude` | Longitude | number | **Ja** | Aus Datenschutzgründen verschobener Längengrad; liegt auf demselben Block wie der tatsächliche Ort. |
| 22 | `location` | Location | location | **Ja** | Kombiniertes Geofeld für Karten und räumliche Operationen; ebenfalls auf Blockebene verschoben. |
| 27 | `:@computed_region_awaf_s7ux` | Historical Wards 2003–2015 | number | Nein | Von Socrata berechnete Zuordnung zu einem historischen Ward von 2003 bis 2015. |
| 28 | `:@computed_region_6mkv_f3dw` | Zip Codes | number | Nein | Von Socrata berechnete Zuordnung zu einem ZIP-Code-Gebiet. |
| 29 | `:@computed_region_vrxf_vc4k` | Community Areas | number | Nein | Von Socrata berechnete Zuordnung zu einer Community Area. |
| 30 | `:@computed_region_bdys_3d7i` | Census Tracts | number | Nein | Von Socrata berechnete Zuordnung zu einem Census Tract. |
| 31 | `:@computed_region_43wa_7qmu` | Wards | number | Nein | Von Socrata berechnete Zuordnung zu einem Ward. |
| 32 | `:@computed_region_rpca_8um6` | Boundaries – ZIP Codes | number | Nein | Von Socrata berechnete Zuordnung zu einer ZIP-Code-Grenzfläche. |
| 33 | `:@computed_region_d9mm_jgwp` | Police Districts | number | Nein | Von Socrata berechnete Zuordnung zu einem Polizeidistrikt. |
| 34 | `:@computed_region_d3ds_rm58` | Police Beats | number | Nein | Von Socrata berechnete Zuordnung zu einem Police Beat. |
| 35 | `:@computed_region_8hcu_yrd4` | Wards 2023– | number | Nein | Automatisch aus `location` berechnete Zuordnung zu einem seit 2023 gültigen Ward. |

Die Positionsnummern stammen direkt aus den Socrata-Metadaten und sind dort nicht fortlaufend; deshalb folgt auf Position 22 die Position 27.

## Einordnung der berechneten Regionsfelder

Die Felder mit dem Präfix `:@computed_region_` sind keine ursprünglichen CPD-Felder aus CLEAR. Socrata erzeugt sie anhand des Feldes `location`, damit im Portal Regionskarten und räumliche Zuordnungen erstellt werden können. Wegen ihrer technischen Bezeichner und ihrer Abhängigkeit von verknüpften Grenzdatensätzen werden sie im ersten PoC nicht verwendet. Bei einem späteren Einsatz müssen Bedeutung und Gültigkeitszeitraum der jeweiligen Gebietsgrenzen geprüft werden.